How to run the code:

1. Install the required library:
   pip install nltk

2. Download the Brown corpus and the Punkt tokenizer:
   import nltk
   nltk.download("brown")
   nltk.download("punkt")
   nltk.download("punkt_tab")

3. Open the file wsa_red_green.ipynb in Jupyter Notebook or VSCode.

4. Run all cells from top to bottom.
   The notebook will:
   - train the supervised WSD classifier on the Brown corpus
   - evaluate it on a held-out test set
   - output predictions and evaluation statistics
   - apply the trained model to the external file test_text.txt


In [27]:
import nltk
# nltk.download('punkt')
# nltk.download("punkt_tab") -> I used chatGPT to generate this line because sent_tokenize was not working properly
# nltk.download('brown')
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import brown
import random
from statistics import mean
from nltk.metrics import ConfusionMatrix

By looking at the following outputs we understand that sentences are already splitted into tokens.

In [28]:
# import sentences:
sentences = brown.sents()

# take a look at the data
print(sentences[0])
print(len(sentences))
print(type(sentences))

['The', 'Fulton', 'County', 'Grand', 'Jury', 'said', 'Friday', 'an', 'investigation', 'of', "Atlanta's", 'recent', 'primary', 'election', 'produced', '``', 'no', 'evidence', "''", 'that', 'any', 'irregularities', 'took', 'place', '.']
57340
<class 'nltk.corpus.reader.util.ConcatenatedCorpusView'>



For the following code I chose not to use NLTK’s stopword list in order to avoid removing function words that may still carry useful contextual information for word sense disambiguation.

In [29]:
# creation of the context features extraction 
def context_feature_extractor(sent, i):
    return {
        'prev_token1' : sent[i - 1].lower() if i > 0 else 'NO WORD', 
        # it takes the word before the target word and includes a check to avoid accessing positions before the start of the sentence
        'prev_token2' : sent[i - 2].lower() if i > 1 else 'NO WORD',
        'next_token1' : sent[i + 1].lower() if i < len(sent) - 1 else 'NO WORD',
        # same as above, but for the next word.
        'next_token2' : sent[i + 2].lower() if i < len(sent) - 2 else 'NO WORD'
    }

# let's check
for i in range(len(sentences[0])):    
    print(context_feature_extractor(sentences[0], i))

{'prev_token1': 'NO WORD', 'prev_token2': 'NO WORD', 'next_token1': 'fulton', 'next_token2': 'county'}
{'prev_token1': 'the', 'prev_token2': 'NO WORD', 'next_token1': 'county', 'next_token2': 'grand'}
{'prev_token1': 'fulton', 'prev_token2': 'the', 'next_token1': 'grand', 'next_token2': 'jury'}
{'prev_token1': 'county', 'prev_token2': 'fulton', 'next_token1': 'jury', 'next_token2': 'said'}
{'prev_token1': 'grand', 'prev_token2': 'county', 'next_token1': 'said', 'next_token2': 'friday'}
{'prev_token1': 'jury', 'prev_token2': 'grand', 'next_token1': 'friday', 'next_token2': 'an'}
{'prev_token1': 'said', 'prev_token2': 'jury', 'next_token1': 'an', 'next_token2': 'investigation'}
{'prev_token1': 'friday', 'prev_token2': 'said', 'next_token1': 'investigation', 'next_token2': 'of'}
{'prev_token1': 'an', 'prev_token2': 'friday', 'next_token1': 'of', 'next_token2': "atlanta's"}
{'prev_token1': 'investigation', 'prev_token2': 'an', 'next_token1': "atlanta's", 'next_token2': 'recent'}
{'prev_tok

In [30]:
# use a feature extractor to process target tokens ('red' and 'green') in sentences:
featuresets = []
for sent in sentences:
    for i, token in enumerate(sent): # enumerate returns pairs of (index, element)
        if token.lower() in ('red', 'green'):
            featuresets.append((context_feature_extractor(sent, i), token.lower()))

# let's explore our featuresets:
count_red = sum('red' in label for (_, label) in featuresets)
count_green = sum('green' in label for (_, label) in featuresets)
print('red:', count_red, 'green:', count_green) 
print(len(featuresets), '\n', featuresets[0])
  

red: 197 green: 116
313 
 ({'prev_token1': '(', 'prev_token2': 'e.', 'next_token1': ')', 'next_token2': 'berry'}, 'red')


In [16]:
# shuffle the data:
random.seed(30) # for reproducibility
random.shuffle(featuresets)

a train/test split (80% training, 20% test) as required by the assignment:

In [17]:
train_set, test_set = featuresets[:len(featuresets)*80//100], featuresets[len(featuresets)*80//100:]
# train a Naive Bayes classifier:
classifier = nltk.NaiveBayesClassifier.train(train_set)
# evaluate the classifier:
accuracy = nltk.classify.accuracy(classifier, test_set)
print('Accuracy:', accuracy)
print('Training size:', len(train_set))

#  confusion matrix
test_tags = [label for (_, label) in test_set]
gold_tags = [classifier.classify(features) for (features, _) in test_set]
cm = ConfusionMatrix(test_tags, gold_tags)
print(cm)

Accuracy: 0.6507936507936508
Training size: 250
      |  g    |
      |  r    |
      |  e  r |
      |  e  e |
      |  n  d |
------+-------+
green |<15>10 |
  red | 12<26>|
------+-------+
(row = reference; col = test)



For the baseline implementation, the use of `FreqDist` was inspired by a discussion on Stack Overflow:
https://stackoverflow.com/questions/35086440/python-how-to-compute-the-top-x-most-frequently-used-words-in-an-nltk-corpus


In [24]:
train_labels = [label for (_, label) in train_set]
frequency_dist = nltk.FreqDist(train_labels) # get frequency distribution of labels in the training set
most_common_labels = frequency_dist.max() # take the most common label in the training set

baseline_correct = sum(1 for label in test_tags if label == most_common_labels) # count how many times the most common label appears in the test set
baseline_accuracy = baseline_correct / len(test_tags) 
print('most common label:', most_common_labels)
print('Baseline correct:', baseline_correct, 'out of', len(test_tags))    
print('Baseline accuracy:', baseline_accuracy)

most common label: red
Baseline correct: 38 out of 63
Baseline accuracy: 0.6031746031746031


The Naive Bayes classifier achieves a higher accuracy than the most frequent class baseline, indicating that contextual features provide useful information for distinguishing between the two senses. However, the improvement over the baseline is relatively small. This suggests that class frequency is already a strong predictor for this task, and that the contexts in which red and green occur are partially overlapping.

The following code applies the trained model to new data. Since the file `test_text.txt` is not part of the `NLTK corpora` and is not preprocessed, it is first tokenized using NLTK sentence and word tokenization. The tokenization approach is based on the example provided at:
https://www.geeksforgeeks.org/nlp/tokenize-text-using-nltk-python/

In [19]:
# using the model to classify new instances
with open ('test_text.txt', 'r', encoding='utf-8') as f:
    text = f.read()

correct = 0
incorrect = 0
total = 0
model_predictions = [correct, incorrect, total]

sentences = sent_tokenize(text)
for sent in sentences:
    tokens = word_tokenize(sent)
    for i, token in enumerate(tokens):
        if token.lower() in ('red', 'green'):
            features = context_feature_extractor(tokens, i)
            prediction = classifier.classify(features)
            print(f'Token: {token}\tPrediction: {prediction}')
            if token.lower() == prediction:
                correct += 1
            else:
                incorrect += 1
total = correct + incorrect
print(f'Model predictions - Correct: {correct}, Incorrect: {incorrect}, Total: {total}')
print(f'accuracy: {correct/total if total > 0 else 0}')
'''For the external text file, the original token ('red' or 'green')
   is used as a proxy gold label. These statistics are therefore indicative
   and meant only as an additional qualitative evaluation.'''


Token: red	Prediction: red
Token: green	Prediction: red
Token: red	Prediction: red
Token: green	Prediction: red
Token: red	Prediction: green
Token: green	Prediction: red
Token: red	Prediction: red
Token: green	Prediction: red
Token: red	Prediction: red
Token: green	Prediction: green
Token: red	Prediction: red
Token: Green	Prediction: red
Token: red	Prediction: red
Token: green	Prediction: red
Token: red	Prediction: red
Token: green	Prediction: green
Token: red	Prediction: red
Token: green	Prediction: red
Token: red	Prediction: red
Token: Green	Prediction: red
Model predictions - Correct: 11, Incorrect: 9, Total: 20
accuracy: 0.55


"For the external text file, the original token ('red' or 'green')\n   is used as a proxy gold label. These statistics are therefore indicative\n   and meant only as an additional qualitative evaluation."

EXTRA:

For the following code I took a look at: 

https://stackoverflow.com/questions/16379313/how-to-use-the-a-k-fold-cross-validation-in-scikit-with-naive-bayes-classifier-a

https://www.nltk.org/api/nltk.classify.html

In [20]:
# because of the small size of our dataset, we will use cross-validation to evaluate our model:
n_folds = 5
results = []
subset_size = len(featuresets) // n_folds
for i in range(n_folds):
    testing_this_round = featuresets[i * subset_size:][:subset_size]
    training_this_round = featuresets[:i * subset_size] + featuresets[(i + 1) * subset_size:]
    # train a Naive Bayes classifier:
    classifier = nltk.NaiveBayesClassifier.train(training_this_round)
    # evaluate the classifier:
    accuracy = nltk.classify.accuracy(classifier, testing_this_round)
    round = {'Round': i + 1}
    training_size = {'Training size': len(training_this_round)}
    important_features = classifier.most_informative_features(5)
    results.append((round, training_size, accuracy, important_features))

In [22]:
# show results of cross-validation
print('\nCross-validation results:')
for result in results:
    print('round =', result[0], 'training size =', result[1]['Training size'], 'accuracy =', result[2])
print('Average accuracy:', mean(result[2] for result in results))



Cross-validation results:
round = {'Round': 1} training size = 251 accuracy = 0.7580645161290323
round = {'Round': 2} training size = 251 accuracy = 0.6935483870967742
round = {'Round': 3} training size = 251 accuracy = 0.5967741935483871
round = {'Round': 4} training size = 251 accuracy = 0.7258064516129032
round = {'Round': 5} training size = 251 accuracy = 0.6290322580645161
Average accuracy: 0.6806451612903226


Error Analysis and Model Limitations

Although the Naive Bayes classifier slightly outperforms the most frequent sense baseline, its performance remains limited. This can be explained by several factors. First, the contextual environments of "red" and "green" often overlap, reducing the discriminative power of simple bag-of-words features. Second, Naive Bayes assumes independence between features, an assumption that does not hold in natural language, where syntactic and semantic dependencies are structured and interrelated. Additionally, the most frequent class baseline already performs strongly, suggesting class imbalance and frequency effects play a major role. Finally, the model relies on relatively local lexical features and does not capture deeper syntactic or semantic structure, which may be necessary for more accurate word sense disambiguation.